# Ch 7. 데이터 품질이 크기를 이긴다

**"같은 토큰 수면 잘 정제된 데이터가 이긴다" — Phi 시리즈가 증명한 명제를 de-duplication과 품질 필터로 직접 확인한다.**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part2/ch07_data_quality.ipynb)

In [ ]:
# !pip install -q datasets datasketch anthropic

import torch, json, os
from hashlib import md5
print(f"PyTorch: {torch.__version__}")

try:
    import datasketch
    print(f"datasketch: {datasketch.__version__}")
except ImportError:
    print("datasketch not installed — run: pip install datasketch")

## Phi-1의 증명 — 데이터 품질이 크기를 이긴다

| 모델 | 파라미터 | 학습 토큰 | HumanEval |
|---|---:|---:|---:|
| CodeGen-Mono | 16B | 577B | 29.3% |
| **Phi-1** | **1.3B** | **7B** | **50.6%** |

→ **파라미터 12×, 토큰 80× 적음에도 능력은 1.7×.**

**FineWeb-Edu**: 웹 크롤 15T 토큰 → 교육적 가치 점수 ≥ 3인 1.3T만 남김.
데이터를 1/12로 줄였는데 MMLU 점수는 38.7 → 44.1로 향상.

## 최소 예제 — Exact dedup 30초

MD5 해시로 완전히 같은 문서를 제거. 전형적으로 합성 데이터의 5%가 중복.

In [ ]:
# dedup.py — Exact deduplication
from hashlib import md5

# 테스트용 샘플 데이터 생성 (실제로는 tinystories_ko.jsonl)
sample_docs = [
    {"text": "토끼 토토는 당근을 좋아했어요. 어느 날 비가 내렸어요."},
    {"text": "곰 두두는 달을 바라보았어요. 달이 참 예뻤어요."},
    {"text": "토끼 토토는 당근을 좋아했어요. 어느 날 비가 내렸어요."},  # 중복!
    {"text": "고양이 미미는 친구들과 놀았어요."},
    {"text": "고양이 미미는 친구들과 놀았어요."},  # 중복!
    {"text": "작은 마을에 새 한 마리가 노래했어요."},
]

# 샘플 JSONL 파일 생성
with open("tinystories_ko.jsonl", "w") as f:
    for d in sample_docs:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

# Exact dedup 실행
with open("tinystories_ko.jsonl") as f:
    docs = [json.loads(l) for l in f]

# 1. Exact dedup — 완전히 같은 텍스트 제거
seen = set()
out = []
for d in docs:
    h = md5(d["text"].encode()).hexdigest()
    if h in seen: continue
    seen.add(h)
    out.append(d)

print(f"  before: {len(docs)}")
print(f"  after:  {len(out)}  ({(len(docs)-len(out))/len(docs):.1%} 제거)")

# dedup 결과 저장
with open("tinystories_ko.dedup.jsonl", "w") as f:
    for d in out:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

In [ ]:
# 실제 TinyStories에서 100개 다운로드 후 dedup 적용
from datasets import load_dataset
from hashlib import md5

ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
stories = []
for i, row in enumerate(ds):
    if i >= 200: break  # 200개 샘플 (영어판은 중복 거의 없음)
    stories.append({"text": row["text"]})

seen = set()
deduped = []
for d in stories:
    h = md5(d["text"].encode()).hexdigest()
    if h in seen: continue
    seen.add(h)
    deduped.append(d)

print(f"TinyStories 200개 exact dedup:")
print(f"  before: {len(stories)}")
print(f"  after:  {len(deduped)}  ({(len(stories)-len(deduped))/len(stories):.1%} 제거)")

## 실전 — 품질 필터 + Near-dup

### Near-dup (MinHash + LSH)

- **threshold=0.7** — Jaccard similarity 70% 이상이면 중복으로 판정. SmolLM2가 쓴 값.
- 같은 그룹의 첫 문서만 보존.

### LLM judge 품질 점수

- Haiku가 judge로 충분. 5K × 짧은 호출 ≈ $0.5
- **3점 이상** — Phi-3 / FineWeb-Edu가 쓴 임계와 비슷.

In [ ]:
# near_dedup.py — MinHash LSH
from datasketch import MinHash, MinHashLSH
import json

with open("tinystories_ko.dedup.jsonl") as f:
    scored = [json.loads(l) for l in f]

# 약간 비슷한 문서 추가 (near-dup 테스트용)
scored.append({"text": "토끼 토토는 당근을 정말 좋아했어요. 어느 날 비가 내렸어요."})
scored.append({"text": "토끼 토토는 당근을 아주 좋아했어요. 어느 날 비가 많이 내렸어요."})

def shingles(text, n=5):
    """5-gram 글자 단위 shingle."""
    return {text[i:i+n] for i in range(len(text)-n+1)}

lsh = MinHashLSH(threshold=0.7, num_perm=128)  # threshold=0.7
hashes = {}
for i, d in enumerate(scored):
    m = MinHash(num_perm=128)
    for sh in shingles(d["text"]):
        m.update(sh.encode())
    lsh.insert(i, m)
    hashes[i] = m

kept = []
seen_groups = set()
for i, d in enumerate(scored):
    similar = lsh.query(hashes[i])  # 같은 그룹 찾기
    group = min(similar)
    if group in seen_groups: continue
    seen_groups.add(group)
    kept.append(d)

print(f"Near-dup 결과:")
print(f"  입력: {len(scored)}개")
print(f"  출력: {len(kept)}개 ({(len(scored)-len(kept))/len(scored):.1%} 제거)")

In [ ]:
# quality_score.py — LLM judge 품질 점수 (API 필요)
import os

JUDGE_PROMPT = """다음 어린이 동화를 0~5 점으로 평가해줘.

기준:
- 문법: 한국어 자연스러움
- 일관성: 인물·사건의 흐름이 깨지지 않음
- 어휘: 3~5세 어린이에게 적합 (어려운 한자어 X)
- 길이: 200~500자

점수만 출력 (한 자리 정수). 동화:
\"\"\"
{text}
\"\"\""""

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY가 없어 규칙 기반 점수로 대체합니다.")

    def score_rule(text):
        """규칙 기반 품질 점수 (API 없이 사용)."""
        score = 3  # 기본
        if len(text) < 100: score -= 2
        elif len(text) < 150: score -= 1
        if any(w in text for w in ["GPT", "AI", "Claude"]): score -= 2
        if text.count("같은") > 3: score -= 1
        return max(0, min(5, score))

    with open("tinystories_ko.dedup.jsonl") as f:
        docs = [json.loads(l) for l in f]

    scored_docs = []
    for d in docs:
        s = score_rule(d["text"])
        if s >= 3:
            scored_docs.append({**d, "score": s})

    print(f"  규칙 기반 필터 결과: {len(scored_docs)}/{len(docs)} 통과")

else:
    import anthropic
    client = anthropic.Anthropic()

    def score_api(text):
        msg = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=8,
            messages=[{"role": "user", "content": JUDGE_PROMPT.format(text=text)}]
        )
        try:
            return int(msg.content[0].text.strip())
        except:
            return 0

    with open("tinystories_ko.dedup.jsonl") as f:
        docs = [json.loads(l) for l in f]

    scored_docs = []
    for i, d in enumerate(docs):
        s = score_api(d["text"])
        if s >= 3:
            scored_docs.append({**d, "score": s})
        if i % 10 == 0: print(f"  {i}/{len(docs)}, kept {len(scored_docs)}")

    print(f"  filter pass: {len(scored_docs)}/{len(docs)} ({len(scored_docs)/len(docs):.0%})")

In [ ]:
# 토큰 수 산수 — 최종 corpus가 Chinchilla 비율 대비 어느 정도인가
from tokenizers import Tokenizer
import os

if os.path.exists("tokenizer_ko.json"):
    tok = Tokenizer.from_file("tokenizer_ko.json")
elif os.path.exists("tokenizer.json"):
    tok = Tokenizer.from_file("tokenizer.json")
    print("(tokenizer_ko.json 없어 영어 BPE로 대체)")
else:
    print("토크나이저 파일이 없습니다. Ch 6 노트북을 먼저 실행하세요.")
    tok = None

if tok:
    with open("tinystories_ko.dedup.jsonl") as f:
        final_docs = [json.loads(l) for l in f]

    total = sum(len(tok.encode(d["text"]).ids) for d in final_docs)
    print(f"  total tokens: {total/1e6:.2f} M")
    print(f"  for 10M model (Chinchilla 20×): need 200M")
    print(f"  ratio: {total/2e8:.2%}")
    print(f"\n  → 200M 토큰 달성하려면 약 {200e6/total:.0f}배 더 많은 동화 필요")
    print(f"  → TinyStories 영어판(200M+)과 섞으면 해결 가능")

## 연습문제

1. §4의 exact dedup을 본인이 합성한 5K 동화에 적용. 제거율 (%) 측정.
2. §5.1의 judge를 Haiku와 Sonnet 두 모델로 돌려 같은 동화의 점수 차이를 비교. 평균 차이 + 상관계수 (r).
3. **Wikipedia 한국어 1,000 문단**을 다운로드해 §5.2의 near-dup을 적용. 0.5/0.7/0.9 threshold별 제거율은?
4. 본 책 10M 모델에 **TinyStories 영어 50% + 한국어 합성 50%**로 200M 토큰 구성. 각 언어 토큰 수와 동화 수를 산출하라.
5. **(생각해볼 것)** "데이터 품질이 크기를 이긴다"는 명제의 **상한**은 어디일까? 아무리 깨끗한 1M 토큰이라도 70B 모델 학습엔 부족할 것이다. 어디까지가 효과적인가?

In [ ]:
# 연습문제 1 — 이미 위에서 완료. 규모를 늘려 재실행:
# 5K 동화 합성 후 아래 코드 실행
print("5K 동화 합성 후 아래 코드를 실행하세요:")
print("""
with open('tinystories_ko_5k.jsonl') as f:
    docs = [json.loads(l) for l in f]

seen = set()
out = []
for d in docs:
    h = md5(d['text'].encode()).hexdigest()
    if h in seen: continue
    seen.add(h)
    out.append(d)

print(f'제거율: {(len(docs)-len(out))/len(docs):.1%}')
""")

In [ ]:
# 연습문제 3 — Wikipedia 한국어 near-dup (threshold별)
from datasets import load_dataset
from datasketch import MinHash, MinHashLSH

# Wikipedia 한국어 다운로드
try:
    wiki = load_dataset("wikipedia", "20220301.ko", split="train", streaming=True)
    wiki_texts = []
    for i, row in enumerate(wiki):
        if i >= 1000: break
        # 문단 단위로 분리
        paragraphs = [p for p in row["text"].split("\n") if len(p) > 50]
        wiki_texts.extend(paragraphs[:3])  # 각 문서에서 최대 3 문단
        if len(wiki_texts) >= 1000: break
    wiki_texts = wiki_texts[:1000]
    print(f"Wikipedia 한국어 문단: {len(wiki_texts)}개 수집")
except Exception as e:
    print(f"Wikipedia 로드 실패: {e}")
    # 샘플로 대체
    wiki_texts = [f"한국 역사의 {i}번째 단락입니다. 조선시대부터 현재까지의 역사를 다루고 있습니다." for i in range(50)]
    wiki_texts += wiki_texts[:50]  # 중복 추가
    print(f"샘플 데이터 {len(wiki_texts)}개 사용")

def near_dedup(texts, threshold):
    def shingles(text, n=5):
        return {text[i:i+n] for i in range(len(text)-n+1)}

    lsh = MinHashLSH(threshold=threshold, num_perm=64)
    hashes = {}
    for i, text in enumerate(texts):
        m = MinHash(num_perm=64)
        for sh in shingles(text):
            m.update(sh.encode())
        try:
            lsh.insert(i, m)
        except ValueError:
            pass  # 이미 추가된 경우 무시
        hashes[i] = m

    kept = []
    seen_groups = set()
    for i, text in enumerate(texts):
        if i not in hashes: continue
        similar = lsh.query(hashes[i])
        group = min(similar) if similar else i
        if group in seen_groups: continue
        seen_groups.add(group)
        kept.append(text)
    return kept

print(f"\n{'Threshold':>10s}  {'남은 개수':>10s}  {'제거율':>10s}")
print("-" * 36)
for threshold in [0.5, 0.7, 0.9]:
    kept = near_dedup(wiki_texts, threshold)
    removal = (len(wiki_texts) - len(kept)) / len(wiki_texts)
    print(f"  {threshold:>8.1f}  {len(kept):>10d}  {removal:>9.1%}")

In [ ]:
# 연습문제 4 — 200M 토큰 혼합 구성 산출

# 가정:
# TinyStories 영어: 평균 약 200 tokens/story
# 한국어 합성: 평균 약 100 tokens/story (짧은 한국어 동화)

target_total = 200_000_000  # 200M 토큰
en_ratio = 0.5  # 50%
ko_ratio = 0.5  # 50%

en_tokens = target_total * en_ratio
ko_tokens = target_total * ko_ratio

en_avg_tokens = 200  # tokens/story (TinyStories 영어)
ko_avg_tokens = 100  # tokens/story (한국어 합성)

en_stories = en_tokens / en_avg_tokens
ko_stories = ko_tokens / ko_avg_tokens

print(f"목표: {target_total/1e6:.0f}M 토큰 (영어 {en_ratio:.0%} + 한국어 {ko_ratio:.0%})")
print(f"\n영어:")
print(f"  토큰 수: {en_tokens/1e6:.0f}M")
print(f"  동화 수: {en_stories:,.0f}개 (TinyStories에서 {en_stories/2.4e6:.1%})")
print(f"\n한국어:")
print(f"  토큰 수: {ko_tokens/1e6:.0f}M")
print(f"  동화 수: {ko_stories:,.0f}개 합성 필요")
print(f"  비용 (Haiku): ~${ko_stories * 100 / 1_000_000 * 0.25:.0f}")

In [ ]:
# 운영 체크리스트 확인
checklist = [
    ("Exact dedup (md5)",             True),
    ("Near-dup (MinHash, threshold 0.7)", True),
    ("품질 필터 (judge LLM, 임계 ≥3)",  False),  # API 없어서 규칙 기반 사용
    ("PII 마스킹 (Ch 29)",               False),  # 합성 데이터라 해당 없음
    ("평가셋 분리 (1~2%, hash 검증)",    False),  # TODO
    ("토큰 수 산수",                     True),
    ("라이선스 정리",                    False),  # TODO
    ("사람 검수 100건",                   False),  # TODO
]

print("데이터 큐레이션 체크리스트:")
for item, done in checklist:
    status = "[v]" if done else "[ ]"
    print(f"  {status} {item}")

In [ ]:
# 연습문제 5 — 생각해볼 것
print("""
"데이터 품질이 크기를 이긴다"의 상한:

- Phi-1 (1.3B, 7B 토큰)이 16B 모델을 이겼다.
- 하지만 1B 토큰 고품질 데이터가 70B 모델 학습에는 부족하다.

대략적 상한:
- 품질 데이터의 효과 = "같은 양의 저품질보다 좋다"
- 하지만 모델 크기 × 20 이상의 토큰이 필요 (Chinchilla 비율)
- 고품질 1M 토큰이라도 10B 모델에는 충분하지 않다 (비율이 1/10000)

결론:
- 데이터 품질 효과는 '도메인 압축 효율'에 달려 있다.
- 좁은 도메인 (TinyStories)에서는 1M도 강력하다.
- 넓은 도메인 (일반 LLM)에서는 양과 질 모두 필요하다.
""")